In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    make_scorer, recall_score, precision_score, f1_score,
    accuracy_score, classification_report, confusion_matrix
)

# =======================
# Configurações principais
# =======================
DATA_DIR = Path('dados')
CSV_PATHS = {
    1: DATA_DIR / 'dados_modelo1.csv',
    2: DATA_DIR / 'dados_modelo2.csv',
    3: DATA_DIR / 'dados_modelo3.csv',
}
TARGET = 'Aprovou_Aprovou'
RANDOM_STATE = 42

# Classe minoritária e meta de recall
MINORITY_LABEL = 0.0
RECALL_MIN_TARGET = 0.50

# Preferência ao escolher threshold entre candidatos que batem a meta
THRESHOLD_PREFERENCE = 'f1'  # 'precision' ou 'f1'

# Margem de segurança para o threshold final (ajuda recall em teste)
THRESHOLD_SHIFT = 0.05  # final_threshold = max(0, median_threshold - THRESHOLD_SHIFT)

# Parâmetros base do RF
RF_BASE_PARAMS = dict(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight='balanced'
)

# Grade de hiperparâmetros enxuta e eficaz
PARAM_GRID = {
    'rf__max_depth': [3, 4, 6, 8],
    'rf__min_samples_leaf': [3, 5, 10],
    'rf__min_samples_split': [5, 10, 20],
    'rf__max_features': ['sqrt', 0.5, None],
}

# CV estratificada
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Scorer: otimize diretamente o recall da minoria
RECALL_MINORITY_SCORER = make_scorer(recall_score, pos_label=MINORITY_LABEL, zero_division=0)

# =======================
# Funções auxiliares
# =======================
def build_preprocess(X: pd.DataFrame) -> ColumnTransformer:
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()
    return ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), num_cols),
            ('cat', Pipeline([
                ('imp', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore'))
            ]), cat_cols)
        ],
        remainder='drop'
    )

def make_pipeline_for_grid(X: pd.DataFrame) -> Pipeline:
    preprocess = build_preprocess(X)
    rf = RandomForestClassifier(**RF_BASE_PARAMS)
    return Pipeline([('prep', preprocess), ('rf', rf)])

def evaluate_threshold_candidates(proba_min: np.ndarray, y_true: pd.Series,
                                  preference: str = THRESHOLD_PREFERENCE,
                                  recall_target: float = RECALL_MIN_TARGET):
    thresholds = np.linspace(0, 1, 201)
    stats = []
    for thr in thresholds:
        y_pred = np.where(proba_min >= thr, MINORITY_LABEL, 1.0)
        rec = recall_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0)
        prec = precision_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0)
        f1 = f1_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0)
        stats.append((thr, rec, prec, f1))
    feasible = [s for s in stats if s[1] >= recall_target]
    if not feasible:
        # Se ninguém bate a meta, pegue os melhores por recall, desempate por F1 e precisão
        feasible = sorted(stats, key=lambda x: (x[1], x[3], x[2]), reverse=True)[:5]
    if preference == 'precision':
        best = max(feasible, key=lambda x: (x[2], x[1], x[3]))
    else:
        best = max(feasible, key=lambda x: (x[3], x[1], x[2]))
    return best  # (thr, rec, prec, f1)

def choose_threshold_cv_robust(pipe: Pipeline, X_train: pd.DataFrame, y_train: pd.Series,
                               preference: str = THRESHOLD_PREFERENCE,
                               recall_target: float = RECALL_MIN_TARGET,
                               cv_inner: StratifiedKFold = CV,
                               threshold_shift: float = THRESHOLD_SHIFT):
    """
    Encontra thresholds por K-fold no treino e usa a mediana, com um pequeno shift para ganhar recall.
    """
    thresholds = []
    fold_stats = []
    X_train = X_train.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    for tr_idx, va_idx in cv_inner.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        # Ajusta cópia do pipe para o fold
        pipe_fold = Pipeline(pipe.steps)  # shallow copy
        pipe_fold.fit(X_tr, y_tr)
        classes_ = pipe_fold.named_steps['rf'].classes_
        idx_min = list(classes_).index(MINORITY_LABEL)
        proba_min_va = pipe_fold.predict_proba(X_va)[:, idx_min]
        thr, rec, prec, f1 = evaluate_threshold_candidates(proba_min_va, y_va, preference, recall_target)
        thresholds.append(thr)
        fold_stats.append({'thr': thr, 'rec': rec, 'prec': prec, 'f1': f1})
    median_thr = float(np.median(thresholds))
    final_thr = max(0.0, median_thr - threshold_shift)  # aplica margem para aumentar recall
    return final_thr, thresholds, fold_stats

def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'recall_minority': recall_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0),
        'precision_minority': precision_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0),
        'f1_minority': f1_score(y_true, y_pred, pos_label=MINORITY_LABEL, zero_division=0),
    }

def run_grid_search(X_train: pd.DataFrame, y_train: pd.Series):
    pipe = make_pipeline_for_grid(X_train)
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=PARAM_GRID,
        scoring=RECALL_MINORITY_SCORER,  # foco em recall da minoria
        cv=CV,
        n_jobs=-1,
        refit=True,
        verbose=1
    )
    grid.fit(X_train, y_train)
    best_pipe = grid.best_estimator_
    best_params = grid.best_params_
    best_cv_score = grid.best_score_
    print("Best params:", best_params)
    print("Best CV Recall(minority):", f"{best_cv_score:.4f}")
    return best_pipe, best_params, best_cv_score

# =======================
# Execução por tabela
# =======================
all_summaries = []

for table_id, csv_path in CSV_PATHS.items():
    print(f"\n======================")
    print(f"Tabela {table_id}: {csv_path}")
    print(f"======================")

    if not Path(csv_path).exists():
        print(f"Arquivo não encontrado: {csv_path}. Pulando.")
        continue

    df = pd.read_csv(csv_path)

    if TARGET not in df.columns:
        print(f"Coluna alvo '{TARGET}' não encontrada em {csv_path}. Pulando.")
        continue

    y = df[TARGET]
    X = df.drop(columns=[TARGET])

    # Splits estratificados: 70% treino, 20% teste, 10% validação
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
    )
    X_test, X_val, y_test, y_val = train_test_split(
        X_temp, y_temp, test_size=1/3, random_state=RANDOM_STATE, stratify=y_temp
    )
    print(f"Formas: treino={X_train.shape}, teste={X_test.shape}, val={X_val.shape}")

    # 1) Grid search maximizando recall da minoria
    best_pipe, best_params, best_cv_recall = run_grid_search(X_train, y_train)

    # 2) Escolha de threshold robusta via CV (no treino)
    thr_cv, thr_list, fold_stats = choose_threshold_cv_robust(
        best_pipe, X_train, y_train,
        preference=THRESHOLD_PREFERENCE,
        recall_target=RECALL_MIN_TARGET,
        cv_inner=CV,
        threshold_shift=THRESHOLD_SHIFT
    )

    # 3) Opcional: validar threshold no conjunto de validação “hold-out” (para report)
    classes_ = best_pipe.named_steps['rf'].classes_
    idx_min = list(classes_).index(MINORITY_LABEL)
    proba_min_val = best_pipe.predict_proba(X_val)[:, idx_min]
    # calcular métricas no val para o thr_cv
    y_pred_val = np.where(proba_min_val >= thr_cv, MINORITY_LABEL, 1.0)
    val_rec = recall_score(y_val, y_pred_val, pos_label=MINORITY_LABEL, zero_division=0)
    val_prec = precision_score(y_val, y_pred_val, pos_label=MINORITY_LABEL, zero_division=0)
    val_f1 = f1_score(y_val, y_pred_val, pos_label=MINORITY_LABEL, zero_division=0)

    # 4) Avaliação no teste usando o threshold robusto
    proba_min_test = best_pipe.predict_proba(X_test)[:, idx_min]
    y_pred_test = np.where(proba_min_test >= thr_cv, MINORITY_LABEL, 1.0)
    metrics = compute_metrics(y_test, y_pred_test)

    print(f"\n[Tabela {table_id}] thr_cv={thr_cv:.3f} (mediana folds={np.median(thr_list):.3f}, shift={THRESHOLD_SHIFT})")
    print(f"Val (thr_cv): recall_min={val_rec:.3f}, precision_min={val_prec:.3f}, f1_min={val_f1:.3f}")
    print(f"Teste: Acc={metrics['accuracy']:.4f}, Recall_min={metrics['recall_minority']:.4f}, "
          f"Prec_min={metrics['precision_minority']:.4f}, F1_min={metrics['f1_minority']:.4f}")
    print("Relatório por classe (teste):")
    print(classification_report(y_test, y_pred_test, digits=4, zero_division=0))
    print("Matriz de confusão (teste):")
    print(confusion_matrix(y_test, y_pred_test))

    all_summaries.append({
        'table_id': table_id,
        'thr_cv': thr_cv,
        'median_thr_folds': float(np.median(thr_list)),
        'best_params': best_params,
        'best_cv_recall_minority': best_cv_recall,
        'val_recall_min_thr': val_rec,
        'val_precision_min_thr': val_prec,
        'val_f1_min_thr': val_f1,
        **metrics
    })

# Resumo geral
if all_summaries:
    df_all = pd.DataFrame(all_summaries).sort_values(['table_id'])
    print("\n======================")
    print("Resumo geral (por tabela):")
    cols_to_show = [
        'table_id','thr_cv','median_thr_folds','best_params','best_cv_recall_minority',
        'val_recall_min_thr','val_precision_min_thr','val_f1_min_thr',
        'accuracy','recall_minority','precision_minority','f1_minority'
    ]
    print(df_all[cols_to_show])
else:
    print("\nNenhum resumo gerado. Verifique os arquivos e a coluna alvo.")


Tabela 1: dados\dados_modelo1.csv
Formas: treino=(962, 10), teste=(275, 10), val=(138, 10)
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best params: {'rf__max_depth': 3, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 10, 'rf__min_samples_split': 5}
Best CV Recall(minority): 0.8727

[Tabela 1] thr_cv=0.520 (mediana folds=0.570, shift=0.05)
Val (thr_cv): recall_min=1.000, precision_min=0.444, f1_min=0.615
Teste: Acc=0.8764, Recall_min=0.7500, Prec_min=0.2857, F1_min=0.4138
Relatório por classe (teste):
              precision    recall  f1-score   support

         0.0     0.2857    0.7500    0.4138        16
         1.0     0.9828    0.8842    0.9309       259

    accuracy                         0.8764       275
   macro avg     0.6343    0.8171    0.6723       275
weighted avg     0.9423    0.8764    0.9008       275

Matriz de confusão (teste):
[[ 12   4]
 [ 30 229]]

Tabela 2: dados\dados_modelo2.csv
Formas: treino=(962, 13), teste=(275, 13), val=(138, 13)
